# 10 — Dense Experiment Analysis (n=1..25)

Reads `results/dense_experiment_results.csv` from `09_classify_dense.py`
and produces:

1. **Efficiency line charts** — Sensitivity / Precision / ROC-AUC / F1 / PR-AUC vs n_coeffs
2. **Top configurations scatter** — Precision vs Sensitivity with article baseline
3. **Chebyshev vs Legendre** — Pairwise Wilcoxon tests (Holm-corrected)
4. **vs OG baseline** — Wilcoxon on repeat-level means
5. **Optimal dimensionality** — Smallest n_coeffs without significant improvement
6. **Best model summary table**
7. **Relative gain curves** — % gain vs OG for ROC-AUC and F1
8. **1-SE near-optimal regions** — ROC-AUC-based 1-SE rule

**Article baseline** (CNN Binary, Youden): Sensitivity = 0.878, Precision = 0.838, ROC-AUC = 0.961

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

ART_SENS = 0.878
ART_PREC = 0.838
ART_ROC = 0.961

EXPERIMENT_DIR = Path.cwd() if Path("results").exists() else Path("transformation_experiment")
RESULTS_DIR = EXPERIMENT_DIR / "results"
MODELS_DIR = EXPERIMENT_DIR / "models_dense"

In [ ]:
df = pd.read_csv(RESULTS_DIR / "dense_experiment_results.csv")
print(f"Loaded {len(df)} result rows")
print(f"Representations: {df['representation'].nunique()}")
print(f"Classifiers: {df['classifier'].unique().tolist()}")
print(f"Splits: {df['split'].nunique()}")
df.head()

In [ ]:
df["repeat"] = df["split"].str.extract(r"rep(\d+)").astype(int)
df["fold"] = df["split"].str.extract(r"fold(\d+)").astype(int)

metric_cols = [
    "roc_auc", "pr_auc", "youden_j", "f1",
    "sensitivity", "specificity", "precision", "accuracy",
    "brier", "log_loss",
]

repeat_means = (
    df.groupby(["representation", "n_features", "classifier", "repeat"])[metric_cols]
    .mean()
    .reset_index()
)

summary = (
    repeat_means
    .groupby(["representation", "n_features", "classifier"])[metric_cols]
    .agg(["mean", "std"])
)
summary.columns = [f"{col}_{stat}" for col, stat in summary.columns]
summary = summary.reset_index().sort_values("f1_mean", ascending=False)

print(f"Repeat-level means: {len(repeat_means)} rows")
print(f"Summary: {len(summary)} configurations")
summary.head(10)

In [ ]:
def parse_repr(name):
    if name == "og_xp_110":
        return "og_xp", 110
    parts = name.rsplit("_", 2)
    return parts[0], int(parts[1])

summary[["basis", "n_coeffs"]] = summary["representation"].apply(
    lambda x: pd.Series(parse_repr(x))
)
repeat_means[["basis", "n_coeffs"]] = repeat_means["representation"].apply(
    lambda x: pd.Series(parse_repr(x))
)

summary = summary.sort_values("f1_mean", ascending=False).reset_index(drop=True)
summary.head(10)

## 1. Efficiency line charts

In [ ]:
CLASSIFIERS = summary["classifier"].unique()
BASES = ["chebyshev", "legendre"]
BASE_COLORS = {"chebyshev": "#2196F3", "legendre": "#FF9800"}
BASE_DASH = {"chebyshev": "solid", "legendre": "dash"}

og_summary = summary[summary["basis"] == "og_xp"]

ROW_HEIGHT = 350


def plot_efficiency(metric, ylabel, title_suffix="", art_ref=None):
    n_clf = len(CLASSIFIERS)
    fig = make_subplots(
        rows=n_clf, cols=1,
        subplot_titles=[str(c) for c in CLASSIFIERS],
        shared_xaxes=True,
        vertical_spacing=0.06,
    )

    all_y_vals = []

    for i, clf in enumerate(CLASSIFIERS):
        row = i + 1
        show_legend = (i == 0)

        for basis in BASES:
            mask = (summary["basis"] == basis) & (summary["classifier"] == clf)
            sub = summary[mask].sort_values("n_coeffs")
            if sub.empty:
                continue

            x = sub["n_coeffs"].values
            y_mean = sub[f"{metric}_mean"].values
            y_std = sub[f"{metric}_std"].values
            all_y_vals.extend(y_mean + y_std)
            all_y_vals.extend(y_mean - y_std)

            fig.add_trace(go.Scatter(
                x=x, y=y_mean,
                mode="lines+markers",
                name=basis.capitalize(),
                legendgroup=basis,
                showlegend=show_legend,
                line=dict(color=BASE_COLORS[basis], dash=BASE_DASH[basis]),
                marker=dict(size=5),
                hovertemplate=f"{basis.capitalize()}<br>"
                              f"n_coeffs=%{{x}}<br>{metric}=%{{y:.4f}}<extra></extra>",
            ), row=row, col=1)

            r, g, b = int(BASE_COLORS[basis][1:3], 16), int(BASE_COLORS[basis][3:5], 16), int(BASE_COLORS[basis][5:7], 16)
            fig.add_trace(go.Scatter(
                x=np.concatenate([x, x[::-1]]),
                y=np.concatenate([y_mean + y_std, (y_mean - y_std)[::-1]]),
                fill="toself",
                fillcolor=f"rgba({r},{g},{b},0.15)",
                mode="lines",
                line=dict(width=0),
                showlegend=False,
                hoverinfo="skip",
            ), row=row, col=1)

        og_row = og_summary[og_summary["classifier"] == clf]
        if not og_row.empty:
            og_val = og_row[f"{metric}_mean"].values[0]
            all_y_vals.append(og_val)
            fig.add_hline(
                y=og_val, row=row, col=1,
                line=dict(color="#4CAF50", dash="dot", width=1.5),
                annotation_text=f"OG XP 110d ({og_val:.3f})",
                annotation_position="bottom right",
                annotation_font_size=10,
            )

        if art_ref is not None:
            all_y_vals.append(art_ref)
            fig.add_hline(
                y=art_ref, row=row, col=1,
                line=dict(color="#E91E63", dash="dashdot", width=1.5),
                annotation_text=f"Article CNN ({art_ref:.3f})",
                annotation_position="top right",
                annotation_font_size=10,
            )

        fig.update_yaxes(title_text=ylabel, row=row, col=1)

    y_lo, y_hi = min(all_y_vals), max(all_y_vals)
    pad = (y_hi - y_lo) * 0.05
    fig.update_yaxes(range=[y_lo - pad, y_hi + pad])

    fig.update_xaxes(
        title_text="Number of coefficients", row=n_clf, col=1,
        tickvals=list(range(1, 26)),
        dtick=1,
    )
    fig.update_layout(
        title_text=f"{ylabel} vs Number of Coefficients{title_suffix}",
        height=ROW_HEIGHT * n_clf,
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    )
    return fig

In [ ]:
fig_sens = plot_efficiency("sensitivity", "Sensitivity", art_ref=ART_SENS)
try:
    fig_sens.write_image(str(RESULTS_DIR / "dense_efficiency_sensitivity.png"), scale=2)
except Exception as e:
    print(f"Static export skipped: {e}")
fig_sens.show()

In [ ]:
fig_prec = plot_efficiency("precision", "Precision", art_ref=ART_PREC)
try:
    fig_prec.write_image(str(RESULTS_DIR / "dense_efficiency_precision.png"), scale=2)
except Exception as e:
    print(f"Static export skipped: {e}")
fig_prec.show()

In [ ]:
fig_auc = plot_efficiency("roc_auc", "ROC-AUC", art_ref=ART_ROC)
try:
    fig_auc.write_image(str(RESULTS_DIR / "dense_efficiency_roc_auc.png"), scale=2)
except Exception as e:
    print(f"Static export skipped: {e}")
fig_auc.show()

In [ ]:
fig_f1 = plot_efficiency("f1", "F1 Score")
try:
    fig_f1.write_image(str(RESULTS_DIR / "dense_efficiency_f1.png"), scale=2)
except Exception as e:
    print(f"Static export skipped: {e}")
fig_f1.show()

In [ ]:
fig_prauc = plot_efficiency("pr_auc", "PR-AUC")
try:
    fig_prauc.write_image(str(RESULTS_DIR / "dense_efficiency_pr_auc.png"), scale=2)
except Exception as e:
    print(f"Static export skipped: {e}")
fig_prauc.show()

## 2. Top configurations scatter

In [ ]:
top_k = summary.sort_values("f1_mean", ascending=False).head(10).copy()

CLF_SYMBOLS = {
    "LogisticRegression": "circle",
    "SVM_RBF": "square",
    "RandomForest": "diamond",
    "XGBoost": "triangle-up",
}
BASIS_SCATTER_COLORS = {
    "chebyshev": "#2196F3",
    "legendre": "#FF9800",
    "og_xp": "#4CAF50",
}

fig_scatter = go.Figure()

fig_scatter.add_hline(y=ART_SENS, line=dict(color="#E91E63", dash="dash", width=1), opacity=0.6)
fig_scatter.add_vline(x=ART_PREC, line=dict(color="#E91E63", dash="dash", width=1), opacity=0.6)

fig_scatter.add_trace(go.Scatter(
    x=[ART_PREC], y=[ART_SENS],
    mode="markers+text",
    marker=dict(symbol="star", size=16, color="#E91E63"),
    text=[f"Article CNN ({ART_PREC:.3f}, {ART_SENS:.3f})"],
    textposition="top left",
    textfont=dict(size=10),
    name="Article CNN",
    showlegend=True,
))

shown_clf = set()
for _, row in top_k.iterrows():
    basis = row["basis"]
    clf = row["classifier"]
    color = BASIS_SCATTER_COLORS.get(basis, "gray")
    symbol = CLF_SYMBOLS.get(clf, "circle")
    n = int(row["n_coeffs"])
    label = f"{basis[:3].capitalize()}{n}"

    fig_scatter.add_trace(go.Scatter(
        x=[row["precision_mean"]], y=[row["sensitivity_mean"]],
        error_x=dict(type="data", array=[row["precision_std"]], visible=True),
        error_y=dict(type="data", array=[row["sensitivity_std"]], visible=True),
        mode="markers+text",
        marker=dict(symbol=symbol, size=10, color=color, opacity=0.85,
                    line=dict(width=1, color="white")),
        text=[label], textposition="top right", textfont=dict(size=9),
        name=f"{clf} | {basis.capitalize()} {n}",
        legendgroup=clf,
        showlegend=(clf not in shown_clf),
        hovertemplate=(
            f"<b>{clf}</b> — {basis.capitalize()} {n}<br>"
            f"Precision: {row['precision_mean']:.4f} +/- {row['precision_std']:.4f}<br>"
            f"Sensitivity: {row['sensitivity_mean']:.4f} +/- {row['sensitivity_std']:.4f}<br>"
            f"F1: {row['f1_mean']:.4f}"
            "<extra></extra>"
        ),
    ))
    shown_clf.add(clf)

for basis, color in BASIS_SCATTER_COLORS.items():
    fig_scatter.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers",
        marker=dict(size=10, color=color),
        name=basis.capitalize(),
        showlegend=True,
    ))

fig_scatter.update_layout(
    title="Top 10 Models: Precision vs Sensitivity",
    xaxis_title="Precision (mean +/- std)",
    yaxis_title="Sensitivity (mean +/- std)",
    height=700, width=900,
    template="plotly_white",
    legend=dict(font_size=10),
)
try:
    fig_scatter.write_image(str(RESULTS_DIR / "dense_top10_scatter.png"), scale=2)
except Exception as e:
    print(f"Static export skipped: {e}")
fig_scatter.show()

## 3. Chebyshev vs Legendre

In [ ]:
n_coeffs_grid = sorted(repeat_means[repeat_means["basis"] != "og_xp"]["n_coeffs"].unique())
test_metric = "f1"

comparison_rows = []

for clf in CLASSIFIERS:
    for n in n_coeffs_grid:
        cheb = repeat_means[
            (repeat_means["basis"] == "chebyshev") &
            (repeat_means["n_coeffs"] == n) &
            (repeat_means["classifier"] == clf)
        ].sort_values("repeat")[test_metric].values

        leg = repeat_means[
            (repeat_means["basis"] == "legendre") &
            (repeat_means["n_coeffs"] == n) &
            (repeat_means["classifier"] == clf)
        ].sort_values("repeat")[test_metric].values

        if len(cheb) < 10 or len(leg) < 10:
            continue

        diff = cheb - leg
        try:
            stat, p = wilcoxon(diff, alternative="two-sided")
        except ValueError:
            stat, p = np.nan, 1.0

        comparison_rows.append({
            "classifier": clf,
            "n_coeffs": n,
            "cheb_mean": cheb.mean(),
            "leg_mean": leg.mean(),
            "diff_mean": diff.mean(),
            "p_value": p,
            "significant": p < 0.05,
        })

df_cheb_vs_leg = pd.DataFrame(comparison_rows)

if len(df_cheb_vs_leg) > 0:
    reject, p_adj, _, _ = multipletests(df_cheb_vs_leg["p_value"], method="holm")
    df_cheb_vs_leg["p_adjusted"] = p_adj
    df_cheb_vs_leg["significant_adj"] = reject

print(f"Chebyshev vs Legendre ({test_metric}, Wilcoxon signed-rank, Holm-corrected):")
df_cheb_vs_leg

## 4. Polynomial vs OG baseline

In [ ]:
og_rows = []

for clf in CLASSIFIERS:
    og_scores = repeat_means[
        (repeat_means["basis"] == "og_xp") &
        (repeat_means["classifier"] == clf)
    ].sort_values("repeat")[test_metric].values

    if len(og_scores) < 10:
        continue

    for basis in BASES:
        for n in n_coeffs_grid:
            poly_scores = repeat_means[
                (repeat_means["basis"] == basis) &
                (repeat_means["n_coeffs"] == n) &
                (repeat_means["classifier"] == clf)
            ].sort_values("repeat")[test_metric].values

            if len(poly_scores) < 10:
                continue

            diff = poly_scores - og_scores
            try:
                stat, p = wilcoxon(diff, alternative="two-sided")
            except ValueError:
                stat, p = np.nan, 1.0

            og_rows.append({
                "classifier": clf,
                "basis": basis,
                "n_coeffs": n,
                "poly_mean": poly_scores.mean(),
                "og_mean": og_scores.mean(),
                "diff_mean": diff.mean(),
                "p_value": p,
            })

df_vs_og = pd.DataFrame(og_rows)

if len(df_vs_og) > 0:
    reject, p_adj, _, _ = multipletests(df_vs_og["p_value"], method="holm")
    df_vs_og["p_adjusted"] = p_adj
    df_vs_og["significant_adj"] = reject

print(f"Polynomial vs OG XP 110d ({test_metric}, Holm-corrected):")
print(f"Configurations NOT significantly worse than OG (p_adj > 0.05):")
not_worse = df_vs_og[(~df_vs_og["significant_adj"]) | (df_vs_og["diff_mean"] >= 0)]
not_worse.sort_values(["classifier", "basis", "n_coeffs"])

## 5. Optimal dimensionality

In [ ]:
optimal_rows = []

for clf in CLASSIFIERS:
    for basis in BASES:
        prev_scores = None
        prev_n = None
        optimal_n = n_coeffs_grid[-1]

        for n in n_coeffs_grid:
            scores = repeat_means[
                (repeat_means["basis"] == basis) &
                (repeat_means["n_coeffs"] == n) &
                (repeat_means["classifier"] == clf)
            ].sort_values("repeat")[test_metric].values

            if prev_scores is not None and len(scores) >= 10:
                diff = scores - prev_scores
                try:
                    _, p = wilcoxon(diff, alternative="greater")
                except ValueError:
                    p = 1.0

                if p > 0.05:
                    optimal_n = prev_n
                    break

            prev_scores = scores
            prev_n = n

        row_data = summary[
            (summary["basis"] == basis) &
            (summary["n_coeffs"] == optimal_n) &
            (summary["classifier"] == clf)
        ]
        if not row_data.empty:
            r = row_data.iloc[0]
            optimal_rows.append({
                "classifier": clf,
                "basis": basis,
                "optimal_n": optimal_n,
                "roc_auc": f"{r['roc_auc_mean']:.4f} +/- {r['roc_auc_std']:.4f}",
                "sensitivity": f"{r['sensitivity_mean']:.4f} +/- {r['sensitivity_std']:.4f}",
                "precision": f"{r['precision_mean']:.4f} +/- {r['precision_std']:.4f}",
                "f1": f"{r['f1_mean']:.4f} +/- {r['f1_std']:.4f}",
            })

df_optimal = pd.DataFrame(optimal_rows)
print("Optimal dimensionality (smallest n where adding more doesn't help):")
df_optimal

## 6. Best model summary

In [ ]:
manifest_path = MODELS_DIR / "model_manifest.json"
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
else:
    manifest = {}

best_rows = []
for clf in CLASSIFIERS:
    clf_data = summary[summary["classifier"] == clf]
    if clf_data.empty:
        continue
    best = clf_data.sort_values("f1_mean", ascending=False).iloc[0]

    repr_name = best["representation"]
    sample_key = f"{repr_name}__{clf}__rep0_fold0.joblib"
    params = manifest.get(sample_key, {}).get("best_params", {})

    beats_sens = "YES" if best["sensitivity_mean"] >= ART_SENS else "no"
    beats_prec = "YES" if best["precision_mean"] >= ART_PREC else "no"

    best_rows.append({
        "Classifier": clf,
        "Representation": repr_name,
        "n_coeffs": int(best["n_coeffs"]),
        "ROC-AUC": f"{best['roc_auc_mean']:.4f} +/- {best['roc_auc_std']:.4f}",
        "Sensitivity": f"{best['sensitivity_mean']:.4f} +/- {best['sensitivity_std']:.4f}",
        "Precision": f"{best['precision_mean']:.4f} +/- {best['precision_std']:.4f}",
        "F1": f"{best['f1_mean']:.4f} +/- {best['f1_std']:.4f}",
        "Beats Art. Sens?": beats_sens,
        "Beats Art. Prec?": beats_prec,
        "Best Params": str(params) if params else "—",
    })

df_best = pd.DataFrame(best_rows)
print(f"Article baseline: Sensitivity={ART_SENS}, Precision={ART_PREC}")
print()
df_best

## 7. Relative gain curves

In [ ]:
GAIN_METRICS = ["roc_auc", "f1"]
GAIN_LABELS = {"roc_auc": "ROC-AUC", "f1": "F1"}

for metric in GAIN_METRICS:
    n_clf = len(CLASSIFIERS)
    fig = make_subplots(
        rows=n_clf, cols=1,
        subplot_titles=[str(c) for c in CLASSIFIERS],
        shared_xaxes=True,
        vertical_spacing=0.08,
    )

    for i, clf in enumerate(CLASSIFIERS):
        row = i + 1
        show_legend = (i == 0)

        og_val = og_summary[og_summary["classifier"] == clf][f"{metric}_mean"].values
        if len(og_val) == 0:
            continue
        og_val = og_val[0]

        for basis in BASES:
            mask = (summary["basis"] == basis) & (summary["classifier"] == clf)
            sub = summary[mask].sort_values("n_coeffs")
            if sub.empty:
                continue

            x = sub["n_coeffs"].values
            y_gain = ((sub[f"{metric}_mean"].values - og_val) / og_val) * 100

            fig.add_trace(go.Scatter(
                x=x, y=y_gain,
                mode="lines+markers",
                name=basis.capitalize(),
                legendgroup=basis,
                showlegend=show_legend,
                line=dict(color=BASE_COLORS[basis], dash=BASE_DASH[basis]),
                marker=dict(size=5),
                hovertemplate=(
                    f"{basis.capitalize()}<br>"
                    f"K=%{{x}}<br>Gain=%{{y:.2f}}%<extra></extra>"
                ),
            ), row=row, col=1)

        fig.add_hline(y=0, line_dash="dot", line_color="rgba(0,0,0,0.3)",
                      annotation_text="OG baseline", row=row, col=1)

    fig.update_layout(
        height=ROW_HEIGHT * n_clf,
        title_text=f"Relative {GAIN_LABELS[metric]} gain vs full OG model (110-d)",
        showlegend=True,
        template="plotly_white",
    )
    fig.update_yaxes(title_text="Gain (%)")
    fig.update_xaxes(title_text="Number of coefficients", row=n_clf, col=1, dtick=1)
    fig.show()

## 8. 1-SE near-optimal regions

In [ ]:
R = repeat_means["repeat"].nunique()
SE_METRIC = "roc_auc"

one_se_rows = []
one_se_summary = []

for clf in CLASSIFIERS:
    for basis in BASES:
        mask = (summary["basis"] == basis) & (summary["classifier"] == clf)
        sub = summary[mask].sort_values("n_coeffs")
        if sub.empty:
            continue

        best_idx = sub[f"{SE_METRIC}_mean"].idxmax()
        best_row = sub.loc[best_idx]
        mu_best = best_row[f"{SE_METRIC}_mean"]
        se_best = best_row[f"{SE_METRIC}_std"] / np.sqrt(R)
        k_best = int(best_row["n_coeffs"])
        threshold = mu_best - se_best

        near_optimal_ks = sub[sub[f"{SE_METRIC}_mean"] >= threshold]["n_coeffs"].values
        k_1se = int(near_optimal_ks.min()) if len(near_optimal_ks) > 0 else k_best

        one_se_summary.append({
            "classifier": clf,
            "basis": basis,
            "K_best": k_best,
            "K_1se": k_1se,
            "mu_best": mu_best,
            "se_best": se_best,
            "threshold": threshold,
            "reduction": f"{110 // k_1se}x (110 -> {k_1se})",
        })

        for _, r in sub.iterrows():
            one_se_rows.append({
                "classifier": clf,
                "basis": basis,
                "n_coeffs": int(r["n_coeffs"]),
                "near_optimal": r[f"{SE_METRIC}_mean"] >= threshold,
            })

df_1se = pd.DataFrame(one_se_rows)
df_1se_summary = pd.DataFrame(one_se_summary)

print(f"1-SE analysis ({SE_METRIC.upper()}, R={R} repeats):")
print()
df_1se_summary[["classifier", "basis", "K_best", "K_1se",
                 "mu_best", "threshold", "reduction"]]

In [ ]:
CLF_ORDER = ["SVM_RBF", "RandomForest", "LogisticRegression", "XGBoost"]
CLF_DISPLAY = {
    "SVM_RBF": "SVM",
    "RandomForest": "Random Forest",
    "LogisticRegression": "Logistic Regression",
    "XGBoost": "XGBoost",
}
CLF_REGION_COLORS = {
    "SVM_RBF": "#E53935",
    "RandomForest": "#43A047",
    "LogisticRegression": "#FB8C00",
    "XGBoost": "#1E88E5",
}

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f"<b>{SE_METRIC.upper()}</b> — Chebyshev stable-optimal region",
        f"<b>{SE_METRIC.upper()}</b> — Legendre stable-optimal region",
    ],
    shared_yaxes=True,
    horizontal_spacing=0.28,
)

ANNOT_X = 28

for col_idx, basis in enumerate(["chebyshev", "legendre"], 1):
    for clf in CLF_ORDER:
        y_label = CLF_DISPLAY[clf]
        color = CLF_REGION_COLORS[clf]

        mask_s = (
            (df_1se_summary["classifier"] == clf)
            & (df_1se_summary["basis"] == basis)
        )
        if not mask_s.any():
            continue
        info = df_1se_summary[mask_s].iloc[0]
        k_best = int(info["K_best"])
        k_1se = int(info["K_1se"])

        near_mask = (
            (df_1se["classifier"] == clf)
            & (df_1se["basis"] == basis)
            & df_1se["near_optimal"]
        )
        near_ks = sorted(df_1se[near_mask]["n_coeffs"].values.astype(int))
        if not near_ks:
            near_ks = [k_1se]
        k_min, k_max = min(near_ks), max(near_ks)

        fig.add_trace(
            go.Scatter(
                x=[k_min, k_max], y=[y_label, y_label],
                mode="lines",
                line=dict(color=color, width=14),
                showlegend=False, hoverinfo="skip",
            ),
            row=1, col=col_idx,
        )

        if k_1se < k_min:
            fig.add_trace(
                go.Scatter(
                    x=[k_1se, k_min], y=[y_label, y_label],
                    mode="lines",
                    line=dict(color=color, width=2, dash="dot"),
                    showlegend=False, hoverinfo="skip",
                ),
                row=1, col=col_idx,
            )

        fig.add_trace(
            go.Scatter(
                x=[k_best], y=[y_label], mode="markers",
                marker=dict(color="#333", size=11, symbol="circle"),
                showlegend=False,
                hovertemplate=f"Best K = {k_best}<extra></extra>",
            ),
            row=1, col=col_idx,
        )

        fig.add_trace(
            go.Scatter(
                x=[k_1se], y=[y_label], mode="markers",
                marker=dict(
                    color="white", size=13, symbol="diamond",
                    line=dict(width=2.5, color="#333"),
                ),
                showlegend=False,
                hovertemplate=f"1-SE K = {k_1se}<extra></extra>",
            ),
            row=1, col=col_idx,
        )

        if k_min != k_max:
            fig.add_annotation(
                x=(k_min + k_max) / 2, y=y_label,
                yshift=-18,
                text=f"<span style='color:{color}'>K {k_min}–{k_max}</span>",
                showarrow=False, font=dict(size=10),
                row=1, col=col_idx,
            )

        pct_fewer = (1 - k_1se / 110) * 100

        k1se_perf = summary[
            (summary["basis"] == basis)
            & (summary["n_coeffs"] == k_1se)
            & (summary["classifier"] == clf)
        ]
        roc_1se = (
            k1se_perf[f"{SE_METRIC}_mean"].values[0]
            if not k1se_perf.empty else info["mu_best"]
        )
        og_roc = og_summary[
            og_summary["classifier"] == clf
        ][f"{SE_METRIC}_mean"].values
        delta_pp = (roc_1se - og_roc[0]) if len(og_roc) > 0 else 0
        direction = "better" if delta_pp >= 0 else "lower"

        fig.add_annotation(
            x=ANNOT_X, y=y_label,
            text=(
                f"<b>K = {k_1se}</b><br>"
                f"{pct_fewer:.1f}% fewer coeff.<br>"
                f"{abs(delta_pp):.3f} {direction}"
            ),
            showarrow=False, xanchor="left", font=dict(size=11),
            row=1, col=col_idx,
        )

fig.add_trace(go.Scatter(
    x=[None], y=[None], mode="markers",
    marker=dict(color="#333", size=10, symbol="circle"),
    name="Best performing K",
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode="markers",
    marker=dict(color="white", size=13, symbol="diamond",
                line=dict(width=2, color="#333")),
    name="Earliest 1-SE-sufficient K",
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode="lines",
    line=dict(color="#9E9E9E", width=10),
    name="Stable 1-SE region containing best K",
), row=1, col=1)

fig.update_layout(
    title_text="<b>Near-optimal coefficient regions under 1-SE criterion</b>",
    height=380, width=1100,
    template="plotly_white",
    legend=dict(
        orientation="h", yanchor="top", y=-0.18,
        xanchor="center", x=0.5,
    ),
    margin=dict(r=30, l=140, t=80, b=80),
)
fig.update_xaxes(
    title_text="Retained coefficient count K",
    range=[0, 30], dtick=5,
)
fig.update_yaxes(
    categoryorder="array",
    categoryarray=list(reversed([CLF_DISPLAY[c] for c in CLF_ORDER])),
)
fig.show()

## 9. Save results

In [ ]:
df_cheb_vs_leg.to_csv(RESULTS_DIR / "dense_cheb_vs_leg.csv", index=False)
df_vs_og.to_csv(RESULTS_DIR / "dense_vs_og.csv", index=False)
df_optimal.to_csv(RESULTS_DIR / "dense_optimal_dims.csv", index=False)
df_best.to_csv(RESULTS_DIR / "dense_best_models.csv", index=False)
df_1se_summary.to_csv(RESULTS_DIR / "dense_1se_summary.csv", index=False)

print("Saved:")
print("  dense_cheb_vs_leg.csv")
print("  dense_vs_og.csv")
print("  dense_optimal_dims.csv")
print("  dense_best_models.csv")
print("  dense_1se_summary.csv")

## Key takeaways

_Run all cells above, then summarise findings here._